# Solar-cell EL image analysis

Use development images only. Set `SOLARSCAN_IMAGE_DIR`,
`SOLARSCAN_SPLIT_DIR`, and `SOLARSCAN_ANNOTATION_DIR` to the extracted
dataset directories before running. This notebook reads IDs from `dev.txt`
only; it does not open or inspect test images.


In [ ]:
from collections import Counter
import hashlib
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.patches import Rectangle

from src.data import downsample, load_annotations, load_image

image_dir = Path(os.environ["SOLARSCAN_IMAGE_DIR"])
annotation_dir = Path(os.environ["SOLARSCAN_ANNOTATION_DIR"])
split_dir = Path(os.environ["SOLARSCAN_SPLIT_DIR"])
dev_file = split_dir / "dev.txt"
if not image_dir.is_dir() or not annotation_dir.is_dir():
    raise FileNotFoundError("Configured image or annotation directory is missing")
image_paths = {
    path.stem: path
    for path in sorted(image_dir.iterdir())
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif"}
}
annotation_paths = {
    path.stem: path
    for path in sorted(annotation_dir.iterdir())
    if path.is_file() and path.suffix.lower() == ".xml"
}
if not image_paths:
    raise ValueError("No images found in SOLARSCAN_IMAGE_DIR")
if not dev_file.is_file():
    raise FileNotFoundError("Expected the locked dev.txt manifest")
dev_ids = [line.strip() for line in dev_file.read_text().splitlines() if line.strip()]
if not dev_ids:
    raise ValueError("The development split is empty")
if len(dev_ids) != len(set(dev_ids)):
    raise ValueError("The development split contains duplicate IDs")
dev_sha256 = hashlib.sha256(dev_file.read_bytes()).hexdigest()
print(f"Development split SHA-256: {dev_sha256}")
missing_ids = sorted(set(dev_ids) - set(image_paths))
if missing_ids:
    raise FileNotFoundError(f"Development images missing: {missing_ids[:5]}")
missing_annotations = sorted(set(dev_ids) - set(annotation_paths))
if missing_annotations:
    raise FileNotFoundError(
        f"Development annotations missing: {missing_annotations[:5]}"
    )
print(f"Development images available: {len(dev_ids)}")


## Dimensions, dtypes, and visual inspection

The seeded sample is drawn exclusively from `dev.txt`. Review the gallery,
then record representative IDs below before running the focused analyses.


In [ ]:
rng = np.random.default_rng(0)
sample_size = min(50, len(dev_ids))
sample_ids = rng.choice(dev_ids, size=sample_size, replace=False).tolist()
records = []
for image_id in sample_ids:
    image = load_image(image_paths[image_id])
    records.append(
        {
            "image_id": image_id,
            "height": image.shape[0],
            "width": image.shape[1],
            "dtype": str(image.dtype),
            "min": int(image.min()),
            "max": int(image.max()),
        }
    )
image_summary = pd.DataFrame(records)
image_summary.describe(include="all")


In [ ]:
columns = 4
rows = (len(sample_ids) + columns - 1) // columns
fig, axes = plt.subplots(rows, columns, figsize=(12, 3 * rows))
for axis, image_id in zip(np.ravel(axes), sample_ids):
    axis.imshow(load_image(image_paths[image_id]), cmap="gray", vmin=0, vmax=255)
    for class_name, box in load_annotations(annotation_paths[image_id]):
        x_min, y_min, x_max, y_max = box
        axis.add_patch(
            Rectangle(
                (x_min, y_min),
                x_max - x_min + 1,
                y_max - y_min + 1,
                fill=False,
                edgecolor="red",
                linewidth=0.8,
            )
        )
    axis.set_title(image_id)
    axis.axis("off")
for axis in np.ravel(axes)[len(sample_ids):]:
    axis.axis("off")
plt.tight_layout()


In [ ]:
dimension_counts = Counter()
instance_counts = Counter()
for image_id in dev_ids:
    dimension_counts[load_image(image_paths[image_id]).shape] += 1
    annotations = load_annotations(annotation_paths[image_id])
    instance_counts.update(name for name, _ in annotations)

class_scope = {
    "in scope": {"finger", "crack", "black_core", "thick_line", "star_crack"},
    "pending decision": {
        "horizontal_dislocation", "short_circuit", "vertical_dislocation"
    },
    "out of scope": {"printing_error", "corner", "fragment", "scratch"},
}
scope_rows = []
for scope, names in class_scope.items():
    scope_rows.append({
        "scope": scope,
        "classes": len(names),
        "instances": sum(instance_counts[name] for name in names),
    })
print("Image dimensions in dev:", dict(dimension_counts))
print("Annotation instances by class:", dict(sorted(instance_counts.items())))
pd.DataFrame(scope_rows)


## Histograms and illumination

After inspecting the gallery, set `typical_id` and `difficult_id` to
development image IDs. Use the same images for the before/after comparisons.


In [ ]:
typical_id = None
difficult_id = None
selected_ids = [item for item in (typical_id, difficult_id) if item is not None]
if not selected_ids:
    raise ValueError("Set typical_id and/or difficult_id after reviewing the gallery")

fig, axes = plt.subplots(len(selected_ids), 2, figsize=(10, 4 * len(selected_ids)))
axes = np.atleast_2d(axes)
for row, image_id in enumerate(selected_ids):
    image = load_image(image_paths[image_id])
    axes[row, 0].imshow(image, cmap="gray", vmin=0, vmax=255)
    axes[row, 0].set_title(image_id)
    axes[row, 1].hist(image.ravel(), bins=256, range=(0, 256))
    axes[row, 1].set_title("Intensity histogram")
plt.tight_layout()


In [ ]:
illumination_id = selected_ids[0]
illumination_image = load_image(image_paths[illumination_id]).astype(np.float64)
row_profile = illumination_image.mean(axis=1)
column_profile = illumination_image.mean(axis=0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(row_profile)
axes[0].set_title("Mean intensity by row")
axes[1].plot(column_profile)
axes[1].set_title("Mean intensity by column")
plt.tight_layout()


## Centered log-magnitude spectrum

Inspect periodic peaks and note whether their locations align with visible
busbars or grid fingers. Do not infer a notch-filter design from one image alone.


In [ ]:
spectrum_image = load_image(image_paths[illumination_id]).astype(np.float64)
spectrum = np.fft.fftshift(np.fft.fft2(spectrum_image))
log_magnitude = np.log1p(np.abs(spectrum))
plt.figure(figsize=(6, 6))
plt.imshow(log_magnitude, cmap="magma")
plt.title(f"Centered log-magnitude spectrum: {illumination_id}")
plt.colorbar()


## Resolution and gray-level reduction

Compare reduced-resolution images and quantized gray levels on the selected
development image. Record the lowest setting where relevant defects remain
visually distinguishable; this is a qualitative study, not a test-set result.


In [ ]:
study_image = load_image(image_paths[illumination_id])
factors = [1, 2, 4, 8]
fig, axes = plt.subplots(1, len(factors), figsize=(14, 4))
for axis, factor in zip(axes, factors):
    reduced = downsample(study_image, factor)
    axis.imshow(reduced, cmap="gray", vmin=0, vmax=255)
    axis.set_title(f"Downsample factor {factor}: {reduced.shape}")
    axis.axis("off")
plt.tight_layout()

gray_levels = [2, 4, 8, 16, 32, 64, 256]
fig, axes = plt.subplots(1, len(gray_levels), figsize=(16, 4))
for axis, levels in zip(axes, gray_levels):
    step = 256 // levels
    quantized = (study_image // step) * step
    axis.imshow(quantized, cmap="gray", vmin=0, vmax=255)
    axis.set_title(f"{levels} gray levels")
    axis.axis("off")
plt.tight_layout()


## Findings and design decisions

Findings below are from the fixed development manifest only. The visual
observations use a seeded 50-image gallery and representative images.

- Coverage: all 3,150 dev IDs have matching images and annotations. Sizes: 2,794 are 1024×1024 and 356 are 512×512. The 12 observed labels include 5 in-scope, 3 pending a team decision, and 4 documented out of scope.
- Representative images: `img000336` shows a typical cell with visible busbars and defects; `img000127` is an unusually dark, black-core example.
- Illumination: per-image mean intensity has 10th/50th/90th percentiles about 1.13/99.75/128.18. `img000336` has strong row-profile dips at repeated horizontal structures and darker borders.
- Spectrum: `img000336` has strong axis-aligned energy consistent with repeated grid/busbar structure. One image is not enough to choose a notch filter.
- Resolution: downsampling `img000336` by 4 (to 256×256) retains coarse cell/busbar structure; factor 8 visibly softens detail. This is qualitative, not a chosen setting.
- Gray levels: 16 levels retain broad structure; 8 or fewer show visible banding and detail loss. Validate across dev images before selecting a setting.
- Decisions: keep dev-only tuning; quantify resolution/gray-level effects on the dev set; include the five dataset-card-approved classes, and get a team decision on the three pending classes before final metric calculations.
